# Fine-Tuning Olala for Translation with TRL (SFT)

This notebook fine-tunes **Olala 7A1B** (hybrid Mamba-3 / attention, MoE with 256 experts, 6 active) for
English↔French translation with Supervised Fine-Tuning (SFT), [TRL](https://github.com/huggingface/trl) and LoRA.
It trains on central banking / finance sentence pairs from the open ECB corpus and measures BLEU and chrF++
before and after training, on 400 held-out ECB sentences.

## 🎯 What you'll find

- **The task**: a JSON translation request goes in (languages, text, terminology, domain), the translation comes out
- **LoRA + SFT**: train on the translations only, with reasoning disabled
- **Evaluation**: BLEU and chrF++ per direction, before and after training, and side-by-side examples

## ⚙️ Before you start (local version)

This is the Colab notebook adapted to run on your own machine: no Google Drive, and every file goes under `WORK_DIR`.
- **GPU**: a free **A100 or H100** (set `CUDA_VISIBLE_DEVICES` in the first code cell).
- **Python 3.12 in a fresh venv**, registered as a Jupyter kernel. The install cell changes the kernel's environment
  (it pins torch 2.11 and removes torchvision), so don't run it in an environment you use for something else:
  ```bash
  python3.12 -m venv ~/olala-local/mt-venv && source ~/olala-local/mt-venv/bin/activate
  pip install ipykernel && python -m ipykernel install --user --name olala-mt --display-name "Olala MT (py3.12)"
  ```
  then pick the **Olala MT (py3.12)** kernel. The Mamba-3 kernels compile at run time and may need the CUDA 12.8
  toolkit (`nvcc`).
- **The starting checkpoint** is downloaded from Hugging Face into `WORK_DIR` the first time (13.6 GB), and reused after.
- **The data** is read from the repo's `open_data/` folder: run the notebook from the repo (`colab/`).
- **Quick check**: set `NUM_TRAIN = 64` and `MAX_EVAL = 20` in the first code cell for a run of a few minutes.

# 📦 Installation & Setup

We pin every library to the versions Olala was validated with. Olala's Mamba-3 kernels come from the
[`OVHai-LLM/mamba`](https://github.com/OVHai-LLM/mamba) fork, and its experts from [ScatterMoE](https://github.com/shawntan/scattermoe).
The training fixes (a few patches to ScatterMoE) come from this repo, `script/apply_olala_training_fixes.sh`.
Both are pure Python with kernels compiled on first use, so they need no build step. Attention uses
[FlashAttention 2](https://github.com/Dao-AILab/flash-attention), from a prebuilt wheel for torch 2.11 (the one Olala's
environment pins); without it, the model falls back to PyTorch's slower flex attention. This takes about 5 minutes. Set `INSTALL = False` once the environment is set up.

In [ ]:
import os, site, subprocess, sys
from pathlib import Path

# ---- local settings ----
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # the GPU to use (a free one); set before torch is imported
WORK_DIR = os.path.expanduser("~/olala-mt-tuto")       # kernels, checkpoint and outputs go here
REPO_DIR = Path.cwd().parent if Path.cwd().name == "colab" else Path.cwd()   # the repo root (data in open_data/)
print(REPO_DIR)
NUM_TRAIN = None   # first N training pairs, e.g. 64 for a quick check (None: all 4,000)
MAX_EVAL = None    # first N eval sentences, e.g. 20 for a quick check (None: all 400)
INSTALL = True     # False once the environment is set up
os.makedirs(WORK_DIR, exist_ok=True)

def sh(cmd): subprocess.run(cmd, shell=True, check=True)

if INSTALL:
    sh(f"{sys.executable} -m pip uninstall -q -y torchvision torchaudio")
    sh(f"{sys.executable} -m pip install -q torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128")
    sh(f"{sys.executable} -m pip install -q torch==2.11.0 transformers==5.8.0 trl==1.5.1 peft==0.19.1 accelerate==1.14.0 datasets sacrebleu "
       "tilelang==0.1.9 apache-tvm-ffi==0.1.9 torch-c-dlpack-ext==0.1.5 nvidia-cutlass-dsl==4.5.2 cuda-python==12.9.7 "
       "quack-kernels==0.5.0 z3-solver==4.15.4.0 einops==0.8.2 flash-linear-attention==0.5.2 fla-core==0.5.2 ninja")
    sh(f"{sys.executable} -m pip install -q --no-deps https://github.com/erictang000/flash-attention/releases/download/v2.8.3-torch2.11-clean/flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl")
    for repo, commit in [("OVHai-LLM/mamba", "039a089"), ("shawntan/scattermoe", "47b5e15")]:
        d = f"{WORK_DIR}/" + repo.split("/")[1]
        sh(f"[ -d {d} ] || git clone -q --filter=blob:none https://github.com/{repo} {d}; git -C {d} checkout -q {commit}")
    open(f"{site.getsitepackages()[0]}/olala_kernels.pth", "w").write(f"{WORK_DIR}/mamba\n{WORK_DIR}/scattermoe\n")
    sh(f"bash {REPO_DIR}/script/apply_olala_training_fixes.sh --python {sys.executable} --scattermoe {WORK_DIR}/scattermoe")

# Loading the model from Hugging Face 🤗

We download the checkpoint into `WORK_DIR` (skipped for files already there) and read the data from the repo's
`open_data/` folder. Olala ships its modeling code with the checkpoint, so we load it with `trust_remote_code=True`.

In [ ]:
import json
import socket
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

def prefer_ipv4_if_ipv6_is_slow(host="huggingface.co", timeout=3.0):
    """Some networks resolve IPv6 but can't route it: every request then waits for the IPv6 timeout first.
    Try IPv6 once; if it doesn't connect within `timeout` seconds, use IPv4 first in this kernel."""
    try:
        ipv6 = socket.getaddrinfo(host, 443, socket.AF_INET6, socket.SOCK_STREAM)[0][4][0]
    except socket.gaierror:
        return   # no IPv6 address: nothing to fix
    try:
        socket.create_connection((ipv6, 443), timeout=timeout).close()
        return   # IPv6 works
    except OSError:
        pass
    original = getattr(socket, "_original_getaddrinfo", socket.getaddrinfo)   # safe to run the cell twice
    socket._original_getaddrinfo = original
    def ipv4_first(*args, **kwargs):
        infos = original(*args, **kwargs)
        return [i for i in infos if i[0] == socket.AF_INET] or infos
    socket.getaddrinfo = ipv4_first
    print(f"🌐 IPv6 to {host} did not connect within {timeout:.0f}s: using IPv4 for this session")

prefer_ipv4_if_ipv6_is_slow()

OUTPUT_DIR = f"{WORK_DIR}/olala-sft"   # where the LoRA adapter is saved; created if missing

CKPT_REPO = "OVHaiLLM/Olala-v1.0-mini-7A1B"
MODEL_DIR = f"{WORK_DIR}/olala-ckpt"
print(f"⬇️ Downloading {CKPT_REPO}...")
snapshot_download(CKPT_REPO, local_dir=MODEL_DIR)

DATA_DIR = REPO_DIR / "open_data"
assert (DATA_DIR / "ecb_enfr_train.jsonl").exists(), f"no ECB data in {DATA_DIR}: run the notebook from the repo"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
tokenizer.padding_side = "right"   # Olala relies on causality to ignore padding: pad at the end

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, trust_remote_code=True, dtype=torch.bfloat16, device_map="cuda")
print(f"✅ Model loaded: {model.num_parameters():,} parameters")

# 📚 Build the prompts and the datasets

Olala-MT is prompted with a **system prompt** and a **JSON request**, and answers with the translation only.
We use the non-reasoning mode (`reasoning_effort="disabled"`): the model answers right away. TRL reads the
`chat_template_kwargs` column and passes it to the chat template.

In [ ]:
SYSTEM_PROMPT = (
    "You are a professional translation assistant. Your task is to translate text carefully, pay attention to the "
    "following:\n1. Preserve the casing of words.\n2. Maintain the inline tags as-is in the translated text.\n"
    "3. Follow the terminology mappings provided by the user.\n4. The domain of the text.\n"
    "Return only the translation, without any additional explanation."
)

def build_messages(text, src, tgt, domain="General", terms=None, translation=None):
    request = {"source_language": src, "target_language": tgt, "source_text": text,
               "terminology_mapping": [{"source_term": s, "target_term": t} for s, t in (terms or {}).items()],
               "domain": domain}
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": json.dumps(request, ensure_ascii=False, indent=2)}]
    if translation:
        messages.append({"role": "assistant", "content": translation})
    return messages

print(build_messages("Le chiffre d'affaires a augmenté de 5 %.", "French", "English",
                     terms={"chiffre d'affaires": "revenue"})[1]["content"])

**Training data:** `ecb_enfr_train.jsonl`, 4,000 sentence pairs from the [ECB corpus](https://opus.nlpl.eu/ECB-v1.php)
(European Central Bank publications: monetary policy, banking supervision, payments, statistics), alternating
English→French and French→English, with `"domain": "Finance"`. The rows are already in the chat format above.
`open_data/make_ecb_enfr.py` in the repo builds them: it detokenizes the corpus and drops PDF damage, boilerplate and
misaligned pairs.

**Evaluation data:** `ecb_enfr_eval.jsonl`, 200 English→French and 200 French→English pairs, held out by page and
sharing no 8-word run with the training set, so the score measures domain adaptation, not memorized ECB formulas.

In [ ]:
from datasets import Dataset

def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

def to_eval_row(r):   # a `messages` row -> source / reference and their languages and domain
    request = json.loads(r["messages"][1]["content"])
    return {"source_language": request["source_language"], "target_language": request["target_language"],
            "source": request["source_text"], "domain": request["domain"], "reference": r["messages"][2]["content"]}

train_dataset = Dataset.from_list(read_jsonl(f"{DATA_DIR}/ecb_enfr_train.jsonl")[:NUM_TRAIN])
# the eval rows alternate en→fr / fr→en, so the first MAX_EVAL rows keep both directions
eval_dataset = Dataset.from_list([to_eval_row(r) for r in read_jsonl(f"{DATA_DIR}/ecb_enfr_eval.jsonl")][:MAX_EVAL])
print(train_dataset, eval_dataset, sep="\n")
print(train_dataset[0]["messages"][1]["content"])

# 📏 Evaluate the model before training

We translate each evaluation sentence with greedy decoding, one at a time (Olala's cached generation does not
handle padded batches), keep the `final` channel of the answer, and score with [sacreBLEU](https://github.com/mjpost/sacrebleu).

In [ ]:
import sacrebleu
from tqdm.auto import tqdm

@torch.no_grad()
def translate(model, text, src, tgt, **kwargs):
    inputs = tokenizer.apply_chat_template(build_messages(text, src, tgt, **kwargs), add_generation_prompt=True,
                                           reasoning_effort="disabled", return_tensors="pt", return_dict=True).to("cuda")
    output = model.generate(**inputs, max_new_tokens=384, do_sample=False, use_cache=True)
    answer = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:])   # ...final<|content|>TRANSLATION<|channel_end|>
    return answer.split("<|content|>")[-1].split("<|channel_end|>")[0].strip()

def evaluate(model):
    model.eval()
    hyps = [translate(model, r["source"], r["source_language"], r["target_language"], domain=r["domain"])
            for r in tqdm(eval_dataset)]
    scores = {}
    for src, tgt in [("English", "French"), ("French", "English")]:
        idx = [i for i, r in enumerate(eval_dataset) if r["source_language"] == src]
        h, refs = [hyps[i] for i in idx], [[eval_dataset[i]["reference"] for i in idx]]
        scores[f"{src[:2].lower()}→{tgt[:2].lower()}"] = (sacrebleu.corpus_bleu(h, refs).score,
                                                          sacrebleu.corpus_chrf(h, refs, word_order=2).score)
    return hyps, scores

baseline_hyps, baseline = evaluate(model)
for pair, (bleu, chrf) in baseline.items():
    print(f"{pair}: BLEU {bleu:.1f} | chrF++ {chrf:.1f}")

# 🎯 Supervised Fine-Tuning (SFT + LoRA)

## Wrap the model with PEFT

We adapt the large projections: the Mamba-3 input projection (`in_proj`), the attention queries (`c_q`), the mixers'
output projection (`mixer_proj`), and the MoE's shared expert (`fc_1`, `fc_2`) and expert projections (`down_proj`, `up_proj`).
The MoE router stays frozen. The 256 routed experts are fused ScatterMoE tensors, not linear layers, so LoRA does not apply to them.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["in_proj", "c_q", "mixer_proj", "fc_1", "fc_2", "down_proj", "up_proj"],
    task_type="CAUSAL_LM",
)

## Launch training

`assistant_only_loss=True` computes the loss on the assistant turn only: the model learns to write the translation,
not to reproduce the prompt. One epoch over the 4,000 pairs is 250 steps (fewer with `NUM_TRAIN`). The first steps are slow while the kernels
compile; the run takes about 30–45 minutes on an A100.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=f"{WORK_DIR}/olala-mt-sft",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # <- effective batch size of 16 sentence pairs
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    max_length=768,
    assistant_only_loss=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

print("🚀 Starting SFT training...")
trainer.train()
print("🎉 SFT training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

# 📊 Evaluate the fine-tuned model

In [ ]:
finetuned_hyps, finetuned = evaluate(trainer.model)

print(f"{'':<8}{'BLEU before':>13}{'BLEU after':>12}{'chrF++ before':>15}{'chrF++ after':>14}")
for pair in baseline:
    print(f"{pair:<8}{baseline[pair][0]:>13.1f}{finetuned[pair][0]:>12.1f}{baseline[pair][1]:>15.1f}{finetuned[pair][1]:>14.1f}")

# 🔍 Original vs fine-tuned model

In [ ]:
for i in [i for i in [0, 1, 100, 101] if i < len(eval_dataset)]:
    r = eval_dataset[i]
    print(f"📄 {r['source_language']} → {r['target_language']} ({r['domain']})\n   {r['source']}")
    print(f"   reference:  {r['reference']}\n   original:   {baseline_hyps[i]}\n   fine-tuned: {finetuned_hyps[i]}\n")

# 💾 Save and merge the model

We save the LoRA adapter to `OUTPUT_DIR` (`WORK_DIR/olala-sft/lora`). Merging it into the base weights
gives a standalone checkpoint (13.6 GB) that loads like the original, with `trust_remote_code=True`.

In [ ]:
trainer.save_model(f"{OUTPUT_DIR}/lora")
print(f"💾 LoRA adapter saved to: {OUTPUT_DIR}/lora")

# To also save a merged checkpoint, uncomment:
# merged_model = trainer.model.merge_and_unload()
# merged_model.save_pretrained(f"{WORK_DIR}/olala-mt-merged")
# tokenizer.save_pretrained(f"{WORK_DIR}/olala-mt-merged")

# 🚀 Next steps

- **Your own data**: replace the ECB pairs with sentence pairs from your domain, built with `build_messages` (fill
  `domain` and `terms` for terminology). Clean, in-domain references are what make fine-tuning pay off: on general
  data such as OPUS-100 subtitles, scores barely move, because the base model is already strong there.
- **Serving**: the merged model runs with Olala's vLLM fork, [`OVHai-LLM/vllm`](https://github.com/OVHai-LLM/vllm) (see the model card; use `--mamba-ssm-cache-dtype float32`).